<center>

# Projet Annuel– Système de Recommandation   



**Auteurs : Eudes KODIA et Chenny ISHIMWE**  
**Encadrant : Professeur Fabien Panloup**

Master 2 Data Science – Université d’Angers  
Année universitaire 2025–2026

</center>

# comparaison de quatre approches de recommandation

Cette application interactive a été développée afin d’illustrer expérimentalement les principales approches étudiées dans ce projet :

- **Filtrage collaboratif User-User**
- **Filtrage collaboratif Item-Item**
- **Factorisation matricielle non négative (NMF)**
- **Neural Collaborative Filtering (NCF)**

L’objectif est de comparer ces méthodes sur une matrice utilisateur–item de taille **5 × 6**, avec ajout de bruit et sparsité contrôlée.

## Fonctionnalités principales

L’application permet :

- de simuler la **sparsité** en masquant une partie des notes observées ;
- d’évaluer les performances des modèles via le **RMSE** sur les notes masquées ;
- de visualiser les **recommandations Top-k** pour un utilisateur cible ;
- d’analyser les **similarités User-User et Item-Item** à travers des heatmaps ;
- d’interpréter les facteurs latents appris par **NMF** ;
- d’examiner les embeddings appris par **NCF**.

## Intérêt expérimental

Cette expérimentation permet de mettre en évidence les compromis entre :

- **interprétabilité** ;
- **capacité de généralisation** ;
- **robustesse à la sparsité** ;
- **complexité computationnelle**.

In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF

import torch
import torch.nn as nn
import torch.optim as optim

### Page setup

In [ ]:
st.set_page_config(page_title="RecoSys_Eudes_Chenny", layout="wide")
st.title("Filtrage collaboratif : User-User, Item-Item, NMF et NCF")
st.caption(
    "Comparaison : voisinage (User-User, Item-Item), factorisation (NMF), deep learning (NCF) "
    "avec sparsité contrôlée, RMSE holdout, top-k reco, heatmaps et interprétabilité."
)

### Data 

In [ ]:
users = ["User1", "User2", "User3", "User4", "User5"]
items = ["Item1", "Item2", "Item3", "Item4", "Item5", "Item6"]

R_full = np.array(
    [
        [7, 6, 7, 4, 5, 4],
        [6, 7, np.nan, 4, 3, 4],
        [np.nan, 3, 3, 1, 1, np.nan],
        [1, 2, 2, 3, 3, 4],
        [1, np.nan, 1, 2, 3, 3],
    ],
    dtype=float,
)

### Helpers

In [ ]:
def safe_nanmean(a, axis=None, keepdims=False):
    m = np.nanmean(a, axis=axis, keepdims=keepdims)
    if np.isscalar(m):
        return 0.0 if np.isnan(m) else m
    return np.where(np.isnan(m), 0.0, m)

def clip_rating(x, rmin=1, rmax=7):
    return float(np.clip(x, rmin, rmax))

def rmse(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))) if len(y_true) else np.nan

def cosine_sim(u, v, min_common=2):
    mask = ~np.isnan(u) & ~np.isnan(v)
    if mask.sum() < min_common:
        return 0.0
    a = u[mask]
    b = v[mask]
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb + 1e-9))

def to_df(mat, index=users, columns=items):
    return pd.DataFrame(mat, index=index, columns=columns)

def make_observed_and_holdout(R, mask_rate, seed=0):
    rng = np.random.default_rng(seed)
    R_obs = R.copy()

    observed_positions = np.argwhere(~np.isnan(R_obs))
    n_obs = len(observed_positions)
    n_hide = int(round(mask_rate * n_obs))
    n_hide = max(0, min(n_hide, n_obs))

    holdout = []
    if n_hide > 0:
        hide_idx = rng.choice(n_obs, size=n_hide, replace=False)
        for idx in hide_idx:
            u, i = observed_positions[idx]
            holdout.append((int(u), int(i), float(R_obs[u, i])))
            R_obs[u, i] = np.nan

    return R_obs, holdout

### Predictors: User-User / Item-Item

In [ ]:
def user_user_predict(R, k=3, min_common=2, rating_bounds=(1, 7)):
    rmin, rmax = rating_bounds
    R = R.copy()
    m, n = R.shape

    mu = safe_nanmean(R, axis=1, keepdims=True)
    S = R - mu  # centered

    sims = np.zeros((m, m), dtype=float)
    for u in range(m):
        for v in range(m):
            if u == v:
                sims[u, v] = 1.0
            elif v > u:
                s = cosine_sim(S[u], S[v], min_common=min_common)
                sims[u, v] = s
                sims[v, u] = s

    R_pred = R.copy()
    for u in range(m):
        for i in range(n):
            if np.isnan(R_pred[u, i]):
                candidates = [v for v in range(m) if v != u and not np.isnan(R[v, i])]
                if not candidates:
                    R_pred[u, i] = clip_rating(mu[u, 0], rmin, rmax)
                    continue

                cand_sims = np.array([sims[u, v] for v in candidates], dtype=float)
                cand_vals = np.array([S[v, i] for v in candidates], dtype=float)

                order = np.argsort(np.abs(cand_sims))
                top = order[-min(k, len(order)):]
                w = cand_sims[top]
                x = cand_vals[top]

                pred_centered = float(np.dot(w, x) / (np.sum(np.abs(w)) + 1e-9))
                pred = float(mu[u, 0] + pred_centered)
                R_pred[u, i] = clip_rating(pred, rmin, rmax)

    return R_pred, sims

def item_item_predict(R, k=3, min_common=2, rating_bounds=(1, 7)):
    rmin, rmax = rating_bounds
    R = R.copy()
    m, n = R.shape

    mu = safe_nanmean(R, axis=1, keepdims=True)
    S = R - mu  # adjusted cosine basis

    sims = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                sims[i, j] = 1.0
            elif j > i:
                s = cosine_sim(S[:, i], S[:, j], min_common=min_common)
                sims[i, j] = s
                sims[j, i] = s

    R_pred = R.copy()
    for u in range(m):
        rated = np.where(~np.isnan(R[u]))[0]
        for i in range(n):
            if np.isnan(R_pred[u, i]):
                neigh = [j for j in rated if j != i]
                if not neigh:
                    R_pred[u, i] = clip_rating(mu[u, 0], rmin, rmax)
                    continue

                w = np.array([sims[i, j] for j in neigh], dtype=float)
                x = np.array([R[u, j] for j in neigh], dtype=float)

                order = np.argsort(np.abs(w))
                top = order[-min(k, len(order)):]
                w = w[top]
                x = x[top]

                pred = float(np.dot(w, x) / (np.sum(np.abs(w)) + 1e-9))
                R_pred[u, i] = clip_rating(pred, rmin, rmax)

    return R_pred, sims

### Predictors: NMF

In [ ]:
def nmf_predict(R, n_components=2, max_iter=800, seed=0, rating_bounds=(1, 7)):
    rmin, rmax = rating_bounds
    X = np.nan_to_num(R, nan=0.0)

    model = NMF(
        n_components=n_components,
        init="nndsvd",
        random_state=seed,
        max_iter=max_iter,
    )
    W = model.fit_transform(X)
    H = model.components_
    R_hat = W @ H
    R_hat = np.clip(R_hat, rmin, rmax)
    return R_hat, W, H

### Predictors: NCF (Neural CF)

In [ ]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=8, hidden1=32, hidden2=16):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),
        )

    def forward(self, u, i):
        u_vec = self.user_emb(u)
        i_vec = self.item_emb(i)
        x = torch.cat([u_vec, i_vec], dim=1)
        return self.mlp(x).squeeze(-1)

def train_ncf(R, emb_dim=8, epochs=600, lr=0.01, seed=0, device="cpu"):
    torch.manual_seed(seed)

    X, y = [], []
    for u in range(R.shape[0]):
        for i in range(R.shape[1]):
            if not np.isnan(R[u, i]):
                X.append([u, i])
                y.append(R[u, i])

    X = torch.tensor(X, dtype=torch.long, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device)

    model = NCF(R.shape[0], R.shape[1], emb_dim=emb_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        pred = model(X[:, 0], X[:, 1])
        loss = loss_fn(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    return model

def predict_ncf(model, n_users, n_items, rating_bounds=(1, 7), device="cpu"):
    rmin, rmax = rating_bounds
    R_hat = np.zeros((n_users, n_items), dtype=float)

    model.eval()
    with torch.no_grad():
        for u in range(n_users):
            for i in range(n_items):
                val = model(
                    torch.tensor([u], dtype=torch.long, device=device),
                    torch.tensor([i], dtype=torch.long, device=device),
                ).item()
                R_hat[u, i] = np.clip(val, rmin, rmax)

    return R_hat

### Sidebar

In [ ]:
with st.sidebar:
    st.header("Réglages")

    seed = st.number_input("Seed (reproductible)", min_value=0, max_value=10000, value=42, step=1)

    st.subheader("Sparsité contrôlée")
    mask_rate = st.slider("Taux de masquage (holdout)", 0.0, 0.8, 0.25, 0.05,1)

    st.subheader("Voisinages")
    k_user = st.slider("k (User-User)", 1, 10, 3)
    k_item = st.slider("k (Item-Item)", 1, 10, 3)
    min_common = st.slider("Min co-notes pour similarité", 1, 10, 2)

    st.subheader("NMF")
    n_components = st.slider("Facteurs latents (NMF)", 2, 8, 2)
    max_iter = st.slider("Max iter (NMF)", 200, 2000, 800, 100)

    st.subheader("NCF")
    emb_dim = st.slider("Embedding dim (NCF)", 4, 32, 8, 4)
    epochs_ncf = st.slider("Epochs (NCF)", 100, 2000, 600, 100)
    lr = st.select_slider("Learning rate (NCF)", options=[0.1, 0.05, 0.01, 0.005, 0.001], value=0.01)

    st.subheader("Recommandations")
    user_choice = st.selectbox("Utilisateur cible", users, index=2)
    top_k_reco = st.slider("Top-k reco", 1, 5, 3)

In [ ]:
# Holdout

In [ ]:
R_obs, holdout = make_observed_and_holdout(R_full, mask_rate=mask_rate, seed=seed)

c1, c2 = st.columns([1.25, 1.0])
with c1:
    st.subheader("Matrice utilisée (avec notes masquées)")
    st.dataframe(to_df(R_obs).style.format(precision=2))

with c2:
    st.subheader("Jeu de test interne")
    st.write(f"Notes masquées : **{len(holdout)}**")
    if holdout:
        df_hold = pd.DataFrame(
            [{"user": users[u], "item": items[i], "true": r} for (u, i, r) in holdout]
        )
        st.dataframe(df_hold, hide_index=True)


### Compute models

In [ ]:
device = "cpu"
with st.spinner("Calcul des 4 modèles..."):
    R_user, sim_user = user_user_predict(R_obs, k=k_user, min_common=min_common)
    R_item, sim_item = item_item_predict(R_obs, k=k_item, min_common=min_common)

    R_nmf, W_nmf, H_nmf = nmf_predict(
        R_obs, n_components=n_components, max_iter=max_iter, seed=seed
    )

    ncf_model = train_ncf(
        R_obs, emb_dim=emb_dim, epochs=epochs_ncf, lr=lr, seed=seed, device=device
    )
    R_ncf = predict_ncf(ncf_model, len(users), len(items), device=device)

### RMSE on holdout

In [ ]:
true_vals, pred_user_vals, pred_item_vals, pred_nmf_vals, pred_ncf_vals = [], [], [], [], []

for (u, i, rtrue) in holdout:
    true_vals.append(rtrue)
    pred_user_vals.append(R_user[u, i])
    pred_item_vals.append(R_item[u, i])
    pred_nmf_vals.append(R_nmf[u, i])
    pred_ncf_vals.append(R_ncf[u, i])

rmse_user = rmse(true_vals, pred_user_vals)
rmse_item = rmse(true_vals, pred_item_vals)
rmse_nmf = rmse(true_vals, pred_nmf_vals)
rmse_ncf = rmse(true_vals, pred_ncf_vals)

st.divider()
st.subheader("Comparaison (RMSE sur notes masquées)")

m1, m2, m3, m4, m5 = st.columns(5)
m1.metric("RMSE User-User", f"{rmse_user:.3f}" if not np.isnan(rmse_user) else "—")
m2.metric("RMSE Item-Item", f"{rmse_item:.3f}" if not np.isnan(rmse_item) else "—")
m3.metric("RMSE NMF", f"{rmse_nmf:.3f}" if not np.isnan(rmse_nmf) else "—")
m4.metric("RMSE NCF", f"{rmse_ncf:.3f}" if not np.isnan(rmse_ncf) else "—")
m5.metric("Masquage", f"{int(mask_rate * 100)}%")

### Tabs

In [ ]:
tab1, tab2, tab3, tab4, tab5 = st.tabs(
    ["Matrices", "Top-k reco", "Heatmaps", "NMF factors", "NCF embeddings"]
)

with tab1:
    a, b = st.columns(2)
    c, d = st.columns(2)

    with a:
        st.markdown("### User-User")
        st.dataframe(to_df(R_user).style.format(precision=2))

    with b:
        st.markdown("### Item-Item")
        st.dataframe(to_df(R_item).style.format(precision=2))

    with c:
        st.markdown("### NMF")
        st.dataframe(to_df(R_nmf).style.format(precision=2))

    with d:
        st.markdown("### NCF")
        st.dataframe(to_df(R_ncf).style.format(precision=2))

with tab2:
    st.markdown("### Recommandations pour l’utilisateur cible")
    u = users.index(user_choice)

    candidates = np.where(np.isnan(R_obs[u]))[0]
    if len(candidates) == 0:
        st.info("Cet utilisateur n’a aucun item manquant dans la matrice observée (après masquage).")
    else:
        def topk(R_pred, name):
            scores = [(items[i], float(R_pred[u, i])) for i in candidates]
            scores.sort(key=lambda x: x[1], reverse=True)
            return pd.DataFrame(scores[:top_k_reco], columns=["Item", f"Score ({name})"])

        c1, c2, c3, c4 = st.columns(4)
        with c1:
            st.markdown("#### User-User")
            st.dataframe(topk(R_user, "User-User"), hide_index=True)
        with c2:
            st.markdown("#### Item-Item")
            st.dataframe(topk(R_item, "Item-Item"), hide_index=True)
        with c3:
            st.markdown("#### NMF")
            st.dataframe(topk(R_nmf, "NMF"), hide_index=True)
        with c4:
            st.markdown("#### NCF")
            st.dataframe(topk(R_ncf, "NCF"), hide_index=True)

        st.divider()
        scores = {"User-User": rmse_user, "Item-Item": rmse_item, "NMF": rmse_nmf, "NCF": rmse_ncf}
        valid = {k: v for k, v in scores.items() if not np.isnan(v)}
        if valid:
            best = min(valid, key=valid.get)
            st.success(f"Meilleure RMSE ici : **{best}** (RMSE={valid[best]:.3f})")

with tab3:
    st.markdown("### Heatmaps des similarités (voisinages)")
    st.caption("Plus c’est foncé, plus la similarité est forte.")

    try:
        import plotly.express as px

        df_sim_user = pd.DataFrame(sim_user, index=users, columns=users)
        df_sim_item = pd.DataFrame(sim_item, index=items, columns=items)

        colA, colB = st.columns(2)
        with colA:
            fig1 = px.imshow(df_sim_user.values, x=df_sim_user.columns, y=df_sim_user.index, aspect="auto",
                             title="Similarité User-User (notes centrées)")
            st.plotly_chart(fig1, use_container_width=True)

        with colB:
            fig2 = px.imshow(df_sim_item.values, x=df_sim_item.columns, y=df_sim_item.index, aspect="auto",
                             title="Similarité Item-Item (cosinus ajusté)")
            st.plotly_chart(fig2, use_container_width=True)

    except Exception:
        st.info("Plotly non disponible : affichage en tableau.")
        st.dataframe(pd.DataFrame(sim_user, index=users, columns=users).style.format(precision=2))
        st.dataframe(pd.DataFrame(sim_item, index=items, columns=items).style.format(precision=2))

with tab4:
    st.markdown("### Interprétation NMF (facteurs latents)")
    dfW = pd.DataFrame(W_nmf, index=users, columns=[f"Factor {k+1}" for k in range(W_nmf.shape[1])])
    dfH = pd.DataFrame(H_nmf.T, index=items, columns=[f"Factor {k+1}" for k in range(H_nmf.shape[0])])

    cA, cB = st.columns(2)
    with cA:
        st.markdown("#### Profils utilisateurs (W)")
        st.dataframe(dfW.style.format(precision=3))
        st.bar_chart(dfW)

    with cB:
        st.markdown("#### Profils items (Hᵀ)")
        st.dataframe(dfH.style.format(precision=3))
        st.bar_chart(dfH)

with tab5:
    st.markdown("### Interprétation NCF (embeddings)")
    st.caption("Affiche les vecteurs d’embeddings appris pour utilisateurs/items (moins interprétable que NMF, mais utile).")

    user_emb = ncf_model.user_emb.weight.detach().cpu().numpy()
    item_emb = ncf_model.item_emb.weight.detach().cpu().numpy()

    dfU = pd.DataFrame(user_emb, index=users, columns=[f"dim{d+1}" for d in range(user_emb.shape[1])])
    dfI = pd.DataFrame(item_emb, index=items, columns=[f"dim{d+1}" for d in range(item_emb.shape[1])])

    cA, cB = st.columns(2)
    with cA:
        st.markdown("#### Embeddings utilisateurs")
        st.dataframe(dfU.style.format(precision=3))

    with cB:
        st.markdown("#### Embeddings items")
        st.dataframe(dfI.style.format(precision=3))

In [ ]:
st.divider()
with st.expander("Fonctionnement de l'application"):
    st.write(
        "Cette application permet de comparer plusieurs approches de systèmes de recommandation :\n"
        "- les méthodes de voisinage User-User et Item-Item,\n"
        "- la factorisation de matrices via NMF,\n"
        "- et une approche basée sur l’apprentissage profond avec NCF.\n\n"
        "La sparsité est simulée en masquant une partie des évaluations observées afin "
        "d’évaluer la capacité des modèles à reconstruire les notes manquantes à l’aide du RMSE.\n\n"
        "Les heatmaps illustrent la structure des similarités entre utilisateurs et items, "
        "tandis que les recommandations top-k montrent les prédictions finales. "
        "Les onglets dédiés à NMF et NCF permettent d’analyser les représentations latentes apprises par les modèles."
    )